# EDA: BubblSpace relational passages (Qwen3-30B), v3 vs v4 generator

**Dataset:** [`TheFireHacker/bubblspace-relational-passages-Qwen3-30B`](https://huggingface.co/datasets/TheFireHacker/bubblspace-relational-passages-Qwen3-30B), 222,088 kept passages (CC BY-SA 4.0).

Each passage was written by Qwen3-30B-A3B from a real Wikipedia paragraph, to teach one *semantic relation* (cause–effect, part–whole, comparison …). A second Qwen3 pass acted as a fact-check **judge**: `PASS` → `train` split, `FAIL` → `rejected` config with the judge's critique.

The data came from two versions of the generator, told apart by the **`relation_source`** column:

| `relation_source` | generator | how the relation was picked |
|---|---|---|
| `requested` | v3 (first ~67k rows) | We **assigned** one relation; the model had to use it even if the article didn't support it. The first EDA showed this produced contrived comparison/spatial passages. |
| `chosen_from_3` | v4 (remaining ~155k rows) | The model saw **3 candidate relations** and chose the one that fits the article (`relation_candidates` holds all 3). The judge also fails contrived relations, and stock openings ("People often mix up…") are banned. |

**Question this notebook answers:** did v4 fix the forced-relation problem? The go/no-go number is the **blind contrived rate** in section 5.

Sections: 1 setup · 2 overview (all rows) · 3 quality checks · 4 **v3 vs v4 comparison** · 5 **blind labelling** · 6 summary.

## 1. Setup

### 1.1 Install libraries and log in to Hugging Face
- `datasets` downloads the parquet shards from the Hub; `pandas` / `matplotlib` / `seaborn` are for tables and charts.
- The dataset is public, so login isn't strictly needed, but a token raises the download rate limit. Store it in Colab under **🔑 Secrets → `HF_TOKEN`** (enable notebook access).
- A warning about `bottleneck` after the install is harmless.

In [ ]:
!pip -q install -U datasets huggingface_hub pyarrow pandas matplotlib seaborn

from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))   # reads the token from Colab Secrets
print("logged in")

### 1.2 Load kept + rejected passages
- `train` split = passages the judge **passed** (what goes into training).
- `rejected` config = passages the judge **failed**, with a `critique` column explaining why.
- Both are converted to pandas DataFrames: `df` (kept) and `dfr` (rejected).
- **`relation_source` safety fill:** the earliest v3 shards were written before this column existed. The published parquet already fills it, but if any row is missing it we mark it `requested` (that's what those rows were).
- Expect roughly **222k kept** and a larger rejected set; download is a few hundred MB (≈1–2 min).

In [ ]:
from datasets import load_dataset
import pandas as pd

REPO = "TheFireHacker/bubblspace-relational-passages-Qwen3-30B"

kept = load_dataset(REPO, split="train")
rej = load_dataset(REPO, "rejected", split="train")

df = kept.to_pandas()     # judge PASS
dfr = rej.to_pandas()     # judge FAIL (+ critique)

# make sure every row says which generator produced it
for d in (df, dfr):
    if "relation_source" not in d.columns:
        d["relation_source"] = "requested"
    d["relation_source"] = d["relation_source"].fillna("requested")

print("kept:", len(df), "rejected:", len(dfr))
print("overall pass rate: {:.1%}".format(len(df) / (len(df) + len(dfr))))
print("\nkept by generator:\n", df["relation_source"].value_counts())
print("\nrejected by generator:\n", dfr["relation_source"].value_counts())
print("\nkept columns:", list(df.columns))
print("rejected columns:", list(dfr.columns))
df.head(3)

## 2. Overview (all rows, both generators)

### 2.1 Passage length
- Counts words (whitespace-separated) and characters for every passage.
- The generator dropped anything under 80 words, so `kept under 80 words` should be **0**.
- `approx kept tokens` uses ≈1.3 tokens per English word; the real tokenizer count for the training pool was **65.06M tokens**.

In [ ]:
import re

def word_count(t):
    return len(re.findall(r"\S+", t or ""))

for d in (df, dfr):
    d["n_words"] = d["text"].map(word_count)
    d["n_chars"] = d["text"].map(lambda t: len(t or ""))

print("KEPT words:\n", df["n_words"].describe())
print("\nREJECTED words:\n", dfr["n_words"].describe())
print("\nkept under 80 words:", int((df["n_words"] < 80).sum()))
print("approx kept tokens (words*1.3): {:,}".format(int(df["n_words"].sum() * 1.3)))

### 2.2 Kept vs rejected (bar chart)
A quick picture of how many passages survived the fact-check judge overall.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["kept", "rejected"], [len(df), len(dfr)], color=["#2a9d8f", "#e76f51"])
ax.set_title("Fact-check outcomes (all rows)")
ax.set_ylabel("rows")
for i, v in enumerate([len(df), len(dfr)]):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom")
plt.tight_layout(); plt.show()

### 2.3 Length distributions
If kept and rejected look the same, the judge isn't simply rejecting long or short passages; it's judging content.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
axes[0].hist(df["n_words"], bins=40, color="#2a9d8f", edgecolor="white")
axes[0].set_title("Kept: words per passage"); axes[0].set_xlabel("words")
axes[1].hist(dfr["n_words"], bins=40, color="#e76f51", edgecolor="white")
axes[1].set_title("Rejected: words per passage"); axes[1].set_xlabel("words")
plt.tight_layout(); plt.show()

### 2.4 Pass rate by relation (both generators mixed)
- `pass_rate = kept / (kept + rejected)` for each of the 16 relations.
- In the first EDA (v3 only), **comparison** and **spatial** were lowest; the judge kept catching invented comparisons. Section 4.1 splits this by generator.

In [ ]:
kr = df["relation"].value_counts()
rr = dfr["relation"].value_counts()
pass_rel = pd.DataFrame({"kept": kr, "rejected": rr}).fillna(0)
pass_rel["pass_rate"] = pass_rel["kept"] / (pass_rel["kept"] + pass_rel["rejected"])
pass_rel = pass_rel.sort_values("pass_rate")

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(pass_rel.index, pass_rel["pass_rate"], color="#1d3557")
ax.set_xlim(0, 1); ax.set_xlabel("pass rate"); ax.set_title("Pass rate by relation (all rows)")
plt.tight_layout(); plt.show()
pass_rel.sort_values("pass_rate", ascending=False)

### 2.5 Read one passage
Change `i` to any row number to read a full passage with its format, relation, generator and source article.

In [ ]:
from IPython.display import display, Markdown

i = 0  # change me
row = df.iloc[i]
display(Markdown(
    f"**row {i} | {row['format']} | {row['relation']} | {row['relation_source']} | {row['source_title']}**\n\n"
    + row["text"]
))

### 2.6 The `topic` column
`topic` is only filled for *ungrounded* passages (written from a topic name with no Wikipedia text). Every passage here is grounded, so `topic` is always empty, which is expected. **Pending decision:** drop the column or replace it with a domain label (like section 3.4's).

In [ ]:
blank = df["topic"].isna() | (df["topic"].astype(str).str.strip().isin(["", "None"]))
print("kept rows:", len(df))
print("topic blank/missing:", int(blank.sum()), "({:.1%})".format(blank.mean()))
print(df["topic"].value_counts(dropna=False).head(5))

### 2.7 Reading level and length by format
- `level` = the reader the passage was written for (e.g. "a general adult reader").
- The boxplot shows whether some formats (qa, explainer, …) run longer than others.

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
lvl = df["level"].value_counts()
axes[0].barh(lvl.index, lvl.values, color="#264653")
axes[0].set_title("Kept rows by reading level")
sns.boxplot(data=df, x="format", y="n_words", ax=axes[1], color="#a8dadc")
axes[1].set_title("Word length by format (kept)")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.show()

## 3. Quality checks

### 3.1 Source diversity
Each Wikipedia article should be used only once or twice. If one article produced many passages, the model would over-learn it. `max repeats` should stay small (1–3).

In [ ]:
n_unique = df["source_title"].nunique()
top = df["source_title"].value_counts().head(15)
print(f"unique Wikipedia titles in kept: {n_unique:,} / {len(df):,} rows")
print("max repeats of one title:", int(df["source_title"].value_counts().max()))

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top.index[::-1], top.values[::-1], color="#6d6875")
ax.set_title("Top 15 source titles (kept): should be ~1–3 each"); ax.set_xlabel("rows")
plt.tight_layout(); plt.show()

### 3.2 Pass rate by format, and relation × format heatmaps
- **Format** = the writing style asked for: `explainer`, `confusion` (clearing up a mix-up), `qa`, `process`, `compare_prose`.
- The first heatmap shows the **pass rate** for each relation × format cell (red = judge rejects most). The second shows **how many** kept passages each cell has, so we can spot thin cells.

In [ ]:
kf, rf = df["format"].value_counts(), dfr["format"].value_counts()
pass_fmt = pd.DataFrame({"kept": kf, "rejected": rf}).fillna(0)
pass_fmt["pass_rate"] = pass_fmt["kept"] / (pass_fmt["kept"] + pass_fmt["rejected"])
display(pass_fmt.sort_values("pass_rate", ascending=False))

k = df.groupby(["relation", "format"]).size()
r = dfr.groupby(["relation", "format"]).size()
both = pd.DataFrame({"kept": k, "rej": r}).fillna(0)
both["pass_rate"] = both["kept"] / (both["kept"] + both["rej"])

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
sns.heatmap(both["pass_rate"].unstack("format"), annot=True, fmt=".0%", cmap="RdYlGn",
            vmin=0, vmax=1, ax=axes[0])
axes[0].set_title("Fact-check pass rate: relation × format")
sns.heatmap(both["kept"].unstack("format").fillna(0).astype(int), annot=True, fmt="d",
            cmap="Blues", ax=axes[1])
axes[1].set_title("Kept passages: relation × format")
plt.tight_layout(); plt.show()

### 3.3 Why the judge rejected passages (critique themes)
- Searches each rejection critique for keyword patterns (rough regex labels, not exact).
- One critique can match several themes, so shares add up to more than 100%.
- v4's judge also checks **"contrived relation"**, so that theme should appear mostly in `chosen_from_3` rejects. The second table splits themes by generator.

In [ ]:
themes = {
    "unsupported / not in source": r"not (?:supported|mentioned|stated)|no (?:evidence|mention)|unsupported",
    "fabricated / invented":       r"fabricat|invent|introduces?",
    "contradicts source":          r"contradict",
    "wrong entity / misattribution": r"misattribut|incorrectly (?:attribut|assign|implies)|wrong (?:person|entity)",
    "timeline / order":            r"timeline|chronolog|before|after|preced",
    "numbers":                     r"\bnumber|figure|percent|\d{4}",
    "logic / illogical":           r"illogical|logical|inconsistent|nonsens",
    "contrived / forced relation": r"contriv|forced|artificial|not a natural|doesn't fit|does not fit",
}
crit = dfr["critique"].fillna("").str.lower()
for name, pat in themes.items():
    dfr[name] = crit.str.contains(pat, regex=True)

print(dfr[list(themes)].mean().sort_values(ascending=False).round(3).to_string())
print("\nTheme share by generator:")
display(dfr.groupby("relation_source")[list(themes)].mean().round(2).T)

### 3.4 Rough domain label
- Guesses each passage's subject area (person, place, biology, …) from keywords. It's crude: about a third will be `other`.
- The heatmap shows whether some relations fail more in some domains (e.g. comparison on sports articles).
- This could also become the replacement for the empty `topic` column.
- Takes ≈1 min on 222k rows.

In [ ]:
def guess_domain(text):
    t = text.lower()
    if re.search(r"\b(?:born|died|his career|her career|was elected|served as|played for)\b", t): return "person"
    if re.search(r"\b(?:village|town|city|district|county|province|river|located in)\b", t): return "place"
    if re.search(r"\b(?:species|genus|plant|animal|bird|fish|insect|habitat)\b", t): return "biology"
    if re.search(r"\b(?:album|song|film|novel|band|episode|series)\b", t): return "media"
    if re.search(r"\b(?:war|battle|empire|dynasty|treaty|revolution)\b", t): return "history"
    if re.search(r"\b(?:team|season|league|match|championship|tournament)\b", t): return "sports"
    if re.search(r"\b(?:company|business|market|founded|brand)\b", t): return "business"
    return "other"

df["domain"] = df["text"].map(guess_domain)
dfr["domain"] = dfr["text"].map(guess_domain)
print(df["domain"].value_counts(normalize=True).round(3))

kd = df.groupby(["domain", "relation"]).size()
rd = dfr.groupby(["domain", "relation"]).size()
dom = pd.DataFrame({"kept": kd, "rej": rd}).fillna(0)
dom["pass_rate"] = dom["kept"] / (dom["kept"] + dom["rej"])
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(dom["pass_rate"].unstack("relation"), annot=True, fmt=".0%", cmap="RdYlGn",
            vmin=0, vmax=1, ax=ax)
ax.set_title("Pass rate: domain × relation")
plt.tight_layout(); plt.show()

### 3.5 Browse one format × relation cell
Set `FORMAT`, `RELATION` and `N` to read random kept passages from one cell. Add `GEN` to restrict to one generator (`"requested"` or `"chosen_from_3"`, or `None` for both).

In [ ]:
FORMAT, RELATION, N, GEN = "confusion", "input_output", 5, None   # edit me
sub = df[(df["format"] == FORMAT) & (df["relation"] == RELATION)]
if GEN:
    sub = sub[sub["relation_source"] == GEN]
print(f"{len(sub)} kept passages for {FORMAT} × {RELATION} ({GEN or 'both generators'})")
for _, row in sub.sample(min(N, len(sub)), random_state=0).iterrows():
    display(Markdown(f"---\n**{row['source_title']}** · {row['relation_source']} · {row['level']}\n\n{row['text']}"))

## 4. v3 (`requested`) vs v4 (`chosen_from_3`)

### 4.1 Pass rate and stock openings by generator
- **Pass rate:** v4 may pass *fewer* passages than v3 because its judge is stricter (it also fails contrived relations). Lower here can be a good sign.
- **Stock openings:** the first EDA found ≈15% of passages starting with templates like "People often mix up…" / "What happens if…". v4 banned them in the prompt and filtered them out, so `chosen_from_3` should be **≈0%**.
- The last part lists each generator's most common 4-word openings so you can see the variety directly.

In [ ]:
src = pd.DataFrame({
    "kept": df["relation_source"].value_counts(),
    "rejected": dfr["relation_source"].value_counts(),
})
src["pass_rate"] = (src["kept"] / (src["kept"] + src["rejected"])).round(3)
print(src, "\n")

STOCK = r"^(?:people often|people sometimes|a common confusion|one common confusion|what happens if)"
df["stock_open"] = df["text"].str.lower().str.strip().str.contains(STOCK, regex=True)
print("stock-opening share by generator:")
print(df.groupby("relation_source")["stock_open"].mean().round(3))

for s, g in df.groupby("relation_source"):
    print(f"\n--- top 4-word openings: {s} ---")
    print(g["text"].str.split().str[:4].str.join(" ").value_counts().head(8).to_string())

### 4.2 Relation mix: assigned vs model-chosen
- For each generator: what share of kept passages uses each relation.
- **v3** was assigned relations from a fixed rotation, so its mix is roughly flat (minus judge rejections).
- **v4** picked what fits the article, so the mix should follow Wikipedia's content: more `category_membership`, `part_whole`, `property_attribution`; less `comparison` and `spatial`.
- `change` = v4 share − v3 share. Big negatives are relations v3 was over-forcing.

In [ ]:
mix = pd.crosstab(df["relation"], df["relation_source"], normalize="columns").round(3)
mix["change"] = mix["chosen_from_3"] - mix["requested"]
display(mix.sort_values("change"))

fig, ax = plt.subplots(figsize=(9, 7))
mix[["requested", "chosen_from_3"]].sort_values("chosen_from_3").plot.barh(ax=ax)
ax.set_title("Share of kept passages per relation, by generator"); ax.set_xlabel("share")
plt.tight_layout(); plt.show()

### 4.3 Pick rate: when a relation was offered, how often did v4 choose it?
- Every v4 passage was offered 3 candidate relations (`relation_candidates`) and the model chose one (`relation`).
- `pick_rate = times chosen / times offered`. Choosing at random would give ≈0.33.
- **Below 0.33** = the model tends to *decline* that relation (good for comparison/spatial: it's refusing weak fits). **Above 0.33** = it prefers it when available.
- Note this counts only kept passages; the pick may also be affected by which ones the judge passed.
- `to_list` handles candidates stored as a real list or as a string like `"['a','b','c']"`.

In [ ]:
import ast
import numpy as np

v4 = df[df["relation_source"] == "chosen_from_3"].copy()

def to_list(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, str):
        try:
            return list(ast.literal_eval(x))
        except Exception:
            return [x]
    return list(x)

v4["cands"] = v4["relation_candidates"].map(to_list)
print("v4 rows with 3 candidates:", int((v4["cands"].map(len) == 3).sum()), "of", len(v4))

offered = v4.explode("cands")["cands"].value_counts()
chosen = v4["relation"].value_counts()
pick = pd.DataFrame({"offered": offered, "chosen": chosen}).fillna(0).astype(int)
pick["pick_rate"] = (pick["chosen"] / pick["offered"]).round(3)
display(pick.sort_values("pick_rate"))
print("(random choice ≈ 0.33)")

### 4.4 Side by side: the relations that failed in the first EDA
- For each weak relation (comparison, spatial, possession_transfer, and quantity if present), shows **one v3 passage and one v4 passage**.
- Read them in pairs: v3 should read forced ("X is larger than Y" with no basis), v4 should only use the relation when the article supports it.
- Change `random_state` to see a different pair.

In [ ]:
weak = [r for r in df["relation"].unique()
        if any(k in r for k in ("compar", "quant", "spatial", "possession"))]
print("relations checked:", weak)
for r in weak:
    for s in ["requested", "chosen_from_3"]:
        sub = df[(df["relation"] == r) & (df["relation_source"] == s)]
        if len(sub) == 0:
            continue
        row = sub.sample(1, random_state=3).iloc[0]
        display(Markdown(f"---\n**{r} · {s}** ({len(sub):,} rows) · {row['source_title']}\n\n{row['text'][:900]}"))

## 5. Blind labelling (the go/no-go check)

- Takes **30 random kept passages from each generator** (60 total), shuffles them, and shows them **without saying which generator wrote each one**, so your judgement isn't biased.
- For each, type `n` (natural: the relation fits the article), `c` (contrived: the relation feels forced or invented), `s` skip, or `q` to stop early. Partial runs still count.
- At the end it reveals the contrived rate per generator.
- **Decision rule:** if `chosen_from_3` is clearly lower than `requested` (e.g. ≤ half), v4 fixed the problem and the data is good for 2.5a. If they're similar, tell Claude which relations/formats felt contrived (section 5.1 breaks it down).
- ≈15–20 min for 60 passages.

In [ ]:
N = 30
pool = (pd.concat([g.sample(min(N, len(g)), random_state=7) for _, g in df.groupby("relation_source")])
          .sample(frac=1, random_state=7).reset_index(drop=True))

labels = []
for i, row in pool.iterrows():
    # relation_source is deliberately NOT shown
    display(Markdown(f"### {i+1}/{len(pool)} · `{row['format']}` × `{row['relation']}` · {row['source_title']}\n\n{row['text']}"))
    a = input("natural (n) / contrived (c) / skip (s) / quit (q): ").strip().lower()
    if a == "q":
        break
    if a in ("n", "c"):
        labels.append({"relation_source": row["relation_source"], "relation": row["relation"],
                       "format": row["format"], "domain": row["domain"], "label": a})

lab = pd.DataFrame(labels)
print(f"\nlabelled {len(lab)}")
print(lab.groupby("relation_source")["label"].apply(lambda s: f"{(s=='c').mean():.0%} contrived of {len(s)}"))

### 5.1 Where the contrived ones are
Breaks your labels down by relation, format and domain within each generator, to show *which* relations or formats still feel forced in v4. Small counts, so treat as hints.

In [ ]:
if len(lab):
    for col in ["relation", "format", "domain"]:
        print(f"\n--- contrived rate by {col} ---")
        print(lab.groupby(["relation_source", col])["label"]
                 .apply(lambda s: f"{(s=='c').mean():.0%} of {len(s)}").unstack(0).fillna("").to_string())
else:
    print("no labels yet: run the cell above")

## 6. Summary table
One table with the headline numbers for each generator. Paste it back to Claude along with the section 5 result.

In [ ]:
def summarize(s):
    k = df[df["relation_source"] == s]
    r = dfr[dfr["relation_source"] == s]
    out = {
        "kept_rows": len(k),
        "rejected_rows": len(r),
        "pass_rate": f"{len(k)/max(1, len(k)+len(r)):.1%}",
        "median_words": int(k["n_words"].median()) if len(k) else 0,
        "stock_opening_share": f"{k['stock_open'].mean():.1%}" if len(k) else "",
        "unique_sources": int(k["source_title"].nunique()),
        "relations_used": int(k["relation"].nunique()),
        "comparison_share": f"{(k['relation']=='comparison').mean():.1%}" if len(k) else "",
        "spatial_share": f"{(k['relation']=='spatial').mean():.1%}" if len(k) else "",
    }
    if "lab" in globals() and len(lab):
        l = lab[lab["relation_source"] == s]
        out["blind_contrived_rate"] = f"{(l['label']=='c').mean():.0%} of {len(l)}" if len(l) else ""
    return out

summary = pd.DataFrame({s: summarize(s) for s in ["requested", "chosen_from_3"]})
summary